In [0]:
!pip install kaggle
!pip install kagglehub[pandas-datasets]

In [0]:
import kagglehub, shutil, os, delta, time
from kagglehub import KaggleDatasetAdapter
from pyspark import SparkContext
from datetime import datetime, timedelta
from pyspark.sql.functions import col, row_number
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from pyspark.sql.window import Window

def table_exists(database, table):
    count = (spark.sql(f"SHOW TABLES FROM {database}")
                  .filter(f"database='{database}' AND tableName='{table}'")
                  .count())    
    return count == 1

In [0]:
database = "bronze"
tablename = "customers"
id_field = "idCliente"
timestamp_field = "DtAtualizacao"

In [0]:
# O arquivo CSV não possui schema, então foi passado alguns parâmetros para ele definir. 
df_full = spark.read.format("csv").options(sep=";", header=True).load(f"/Volumes/workspace/upsell/full_load/{tablename}/")
schema = StructType([
    StructField("idCliente", StringType(), True),
    StructField("flEmail", StringType(), True),
    StructField("flTwitch", StringType(), True),
    StructField("flYouTube", StringType(), True),
    StructField("flBlueSky", StringType(), True),
    StructField("flInstagram", StringType(), True),
    StructField("qtdePontos", IntegerType(), True),
    StructField("DtCriacao", TimestampType(), True),
    StructField("DtAtualizacao", TimestampType(), True)
])

# print(schema)

In [0]:
tables_config = [
    {
        "database": "bronze",
        "tablename": "customers",
        "id_field": "idCliente",
        "timestamp_field": "DtAtualizacao",
        "schema": StructType([
            StructField("idCliente", StringType(), True),
            StructField("flEmail", StringType(), True),
            StructField("flTwitch", StringType(), True),
            StructField("flYouTube", StringType(), True),
            StructField("flBlueSky", StringType(), True),
            StructField("flInstagram", StringType(), True),
            StructField("qtdePontos", IntegerType(), True),
            StructField("DtCriacao", TimestampType(), True),
            StructField("DtAtualizacao", TimestampType(), True)
        ])
    },
    {
        "database": "bronze",
        "tablename": "transactions",
        "id_field": "IdTransacao",
        "timestamp_field": "DtCriacao",
        "schema": StructType([
            StructField("idTransacao", StringType(), True),
            StructField("idCliente", StringType(), True),
            StructField("DtCriacao", TimestampType(), True),
            StructField("QtdePontos", IntegerType(), True),
            StructField("DescSistemaOrigem", StringType(), True),
        ])
    },
    {
        "database": "bronze",
        "tablename": "transactions_product",
        "id_field": "idTransacaoProduto",
        #"timestamp_field": "DtAtualizacao",
        "schema": StructType([
            StructField("idTransacaoProduto", StringType(), True),
            StructField("idTransacao", StringType(), True),
            StructField("IdProduto", StringType(), True),
            StructField("QtdeProduto", IntegerType(), True),
            StructField("vlProduto", DoubleType(), True),
        ])
    }
]


In [0]:
if not table_exists(database, tablename):
    print("Tabela não existente, criando tabela...")
    df_full = spark.read.format("csv").options(sep=";", header=True).schema(schema).load(f"/Volumes/workspace/upsell/full_load/{tablename}/")
    (df_full.coalesce(1).write.format("delta").mode("overwrite").saveAsTable(f"{database}.{tablename}"))
else:
    print("Tabela já existente, ignorando a carga completa.")

### Atualização dos dados via Streaming + carga periódica do CDC

In [0]:
def ingest_from_kaggle():
    # Definir destino dos arquivos que serão baixados do kaggle
    cache_path = "/Volumes/workspace/upsell/cdc/kaggle_cache"
    os.environ["KAGGLE_CACHE_FOLDER"] = cache_path
    os.makedirs(cache_path, exist_ok=True)

    # Download do dataset direto no cache dentro do Volume
    kaggle_path = kagglehub.dataset_download("teocalvo/teomewhy-loyalty-system")

    # Diretório de landing — será lido pelo bloco de upsert
    dest_path =  "/Volumes/workspace/upsell/cdc/kaggle"
    os.makedirs(dest_path, exist_ok=True   )

    arquivos_copiados = []

    # Copiando arquivos para o diretório de destino
    for file in os.listdir(kaggle_path):
        if file.startswith("clientes") and file.endswith(".csv"):
            base, ext = os.path.splitext(file)
            timestamp = int(time.time())
            new_name = f"{base}_{timestamp}{ext}"

            full_path = os.path.join(kaggle_path, file)
            dest_file = os.path.join(dest_path, new_name)

            # Cópia via driver, sem staging no DBFS Root.
            shutil.copy(full_path, dest_file)
            arquivos_copiados.append(dest_file)

            if not arquivos_copiados:
                print("Nenhum arquivo 'clientes*.csv' encontrado no dataset.")
            else:
                print(f"\n✅ {len(arquivos_copiados)} arquivo(s) prontos em: {dest_path}")

            return dest_path  
        
landing_path = ingest_from_kaggle()

In [0]:
bronze = delta.DeltaTable.forName(spark, f"{database}.{tablename}")

def upsert(df, deltatable):
    df = (df.withColumn("qtdePontos", col("qtdePontos").cast("int"))
          .withColumn("DtCriacao", col("DtCriacao").cast("timestamp"))
          .withColumn("DtAtualizacao", col("DtAtualizacao").cast("timestamp")))

    # Obter registro mais recente por pessoa
    windowSpec = Window.partitionBy(id_field).orderBy(col(timestamp_field).desc())
    df_cdc = df.withColumn("rn", row_number().over(windowSpec)).filter(col("rn") == 1)

    # Realizando o Merge com a delta table na bronze
    (bronze.alias("b")
                .merge(df_cdc.alias("d"), f"b.{id_field} = d.{id_field}")
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
    )

df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("sep", ";")
    .option("header", True)
    .schema(schema)
    .load("/Volumes/workspace/upsell/cdc/kaggle/")
)

stream = (df_stream.writeStream
          .option("checkpointLocation", f"/Volumes/workspace/upsell/cdc/{tablename}_checkpoint/")
          .foreachBatch(upsert)
          .trigger(availableNow=True)
)

In [0]:
stream.start().awaitTermination()

In [0]:
# Pega o último arquivo baixado do Kaggle e verifica quantas linhas possui ao total.
df_cdc = spark.read.format("csv").options(sep=";", header=True).load(f"/Volumes/workspace/upsell/cdc/kaggle")
ids_kaggle = df_cdc.select(id_field).distinct()

# Contabiliza quantas linhas tem na tabela consolidada Bronze
ids_bronze = spark.table(f"{database}.{tablename}").select(id_field).distinct()

# Verifica quantos ID's estão a menos na  bronze em relação ao Kaggle
ids_novos = ids_kaggle.subtract(ids_bronze)

# Exibindo o resultado da validação
print("CDC IDs", ids_kaggle.count())
print("Bronze IDs:", ids_bronze.count())
print("Novos IDs:", ids_novos.count())
